# SmartBag — Random Forest

Mesmo CSV e mesma divisao da MLP. Produz tres artefatos:

| Arquivo | Vai para | Serve para |
|---|---|---|
| `modelo_smartbag.pkl` | app20 `api/` | inferencia na nuvem, via FastAPI |
| `AIoTRandomForest_micromlgen.hpp` | app21 `device/src/` | a mesma floresta, na borda |
| `AIoTSmartBagScaler.hpp` | app21 `device/src/` | normalizacao, identica a do app22 |

Nao ha dados nem modelo pre-treinados aqui.

## 1. Pacotes


In [ ]:
# Versoes fixas: o .pkl gerado aqui e carregado pela API do app20 com EXATAMENTE
# estas versoes (app20/api/requirements.txt). As duas que precisam casar de verdade
# sao numpy e scikit-learn: e nelas que o formato do .pkl se apoia. Mesmas pinagens
# do app17-7/app18, para o ambiente ser um so na trilha inteira.
# O micromlgen roda com versoes mais novas do sklearn; quem exige o pin e o joblib.load
# do outro lado. Se o Colab pedir "Restart session", reinicie e rode de novo.
%pip install -q matplotlib "numpy==2.1.3" "pandas==2.2.3" "scikit-learn==1.6.1" "joblib==1.5.3" "micromlgen==1.1.28"

In [ ]:
import json, hashlib, zipfile, subprocess
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import joblib
from micromlgen import port

## 2. Abrir e conferir o CSV

O mapa `ENTREGA_OK = 0` e `REVISAR_ENTREGA = 1` vale do CSV ate o firmware: e o
numero que o `predict()` do micromlgen devolve no ESP32, no app21.

In [ ]:
from google.colab import files
arquivos = files.upload()
ARQUIVO = next(iter(arquivos))
df = pd.read_csv(ARQUIVO).sort_values(["rodada", "situacao", "timestamp"]).reset_index(drop=True)

FEATURES = ["temperatura", "umidade", "delta_distancia", "luz", "mov_max", "incl_max"]
CLASSES = ["ENTREGA_OK", "REVISAR_ENTREGA"]   # indice = o codigo: 0 e 1
MAPA = {nome: i for i, nome in enumerate(CLASSES)}

X = df[FEATURES].astype(np.float32)
y = df["target"].map(MAPA)
assert y.notna().all(), "Ha target fora de CLASSES. Confira o CSV; nao converta em 0 silenciosamente."
assert df["device"].nunique() == 1, "Use uma execucao de uma equipe."
print("Mapa de classes:", MAPA)
display(pd.crosstab(df["rodada"], df["target"]))

## 3. Separar por rodada
A última rodada fica no teste, como no app17-7. RF e MLP usam o mesmo CSV e a mesma divisão.


In [ ]:
rodada_teste = df["rodada"].max()
indices_treino = df.index[df["rodada"] != rodada_teste]
indices_teste = df.index[df["rodada"] == rodada_teste]
X_treino, X_teste = X.loc[indices_treino], X.loc[indices_teste]
y_treino, y_teste = y.loc[indices_treino], y.loc[indices_teste]
rodadas_treino = sorted(df.loc[indices_treino, "rodada"].unique().tolist())
rodadas_teste = [int(rodada_teste)]
assert set(y_treino) == set(y_teste) == {0, 1}, "Colete pelo menos duas rodadas completas com as duas classes."
print("Treino:", rodadas_treino, "| Teste:", rodadas_teste)

## 4. Treinar

`StandardScaler` + floresta num Pipeline so, como no app17-7. A arvore nao precisa de
normalizacao — ela compara uma feature por vez com um limiar, e escala nao muda a ordem.
O scaler esta aqui por **simetria com o app22**: o aluno aplica o mesmo
`Scaler::standardize()` nos dois apps embarcados, e o app30 que ele recebe de extensao faz igual.

O que **nao** pode e treinar sem scaler e normalizar no ESP32, ou o contrario: os
limiares gravados no header estariam na escala errada, e a predicao sai errada sem
nenhum aviso.

**21 arvores, numero impar.** O `vote.jinja` do micromlgen desempata com `>` estrito:
num empate, vence sempre `votes[0]` — que aqui e `ENTREGA_OK`, justamente o lado errado
para errar. Com 20 arvores um 10–10 e possivel e silencioso. Com 21, nao existe.

In [ ]:
modelo = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(n_estimators=21, random_state=42),
)
modelo.fit(X_treino, y_treino)
predicoes = modelo.predict(X_teste)

escalador = modelo.named_steps["standardscaler"]
floresta = modelo.named_steps["randomforestclassifier"]
print("Configuracao efetiva:", floresta.get_params())

## 5. Avaliar


In [ ]:
print("Acuracia:", accuracy_score(y_teste, predicoes))
print(classification_report(y_teste, predicoes, labels=[0, 1], target_names=CLASSES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_teste, predicoes, labels=[0, 1], display_labels=CLASSES, cmap="Blues")
plt.show()
display(pd.crosstab(df.loc[indices_teste, "situacao"],
                    pd.Series(predicoes, index=y_teste.index, name="predicao")))

In [ ]:
pd.Series(floresta.feature_importances_, index=FEATURES).sort_values().plot.barh(
    title="Importancia das features")
plt.show()

## 6. Comparar com uma feature
Um corte testa o limiar; uma árvore pequena testa uma feature com vários cortes. Se alguma resolver tudo, revise a coleta. Os resultados anteriores do dataset sintético não são metas desta coleta.


In [ ]:
maioria = DummyClassifier(strategy="most_frequent").fit(X_treino, y_treino)
print("Classe majoritária:", maioria.score(X_teste, y_teste))
resultados = []
for coluna in FEATURES:
    linha = {"feature": coluna}
    for profundidade in [1, 5]:
        simples = DecisionTreeClassifier(max_depth=profundidade, random_state=42)
        simples.fit(X_treino[[coluna]], y_treino)
        linha[f"profundidade_{profundidade}"] = simples.score(X_teste[[coluna]], y_teste)
    resultados.append(linha)
display(pd.DataFrame(resultados))


## 7. Salvar o modelo avaliado


In [ ]:
joblib.dump(modelo, "modelo_smartbag.pkl")
recarregado = joblib.load("modelo_smartbag.pkl")
assert np.array_equal(recarregado.predict(X_teste), predicoes), "O .pkl recarregado nao reproduz o modelo avaliado."

PACOTES = ["scikit-learn", "numpy", "pandas", "joblib", "micromlgen"]
metadados = {
    "features": FEATURES,
    "unidades": ["C", "%", "cm", "RAW (0..4095)", "m/s2", "graus"],
    "classes": CLASSES,
    "mapa_classes": MAPA,        # o app21 le o inteiro que sai do predict()
    "normalizacao": "StandardScaler ajustado so no treino; aplicar uma vez antes do predict",
    "rodadas_treino": rodadas_treino,
    "rodadas_teste": rodadas_teste,
    "csv_sha256": hashlib.sha256(Path(ARQUIVO).read_bytes()).hexdigest(),
    "versoes": {p: version(p) for p in PACOTES},
    "modelo": floresta.get_params(),
}
Path("metadados_rf.json").write_text(json.dumps(metadados, indent=2), encoding="utf-8")
print(json.dumps(metadados["versoes"], indent=2))

## 8. Exportar para C++

Duas pecas, porque o micromlgen porta **so a floresta** — o scaler do Pipeline fica de
fora e precisa virar header a parte. No ESP32 a ordem e: ler os seis valores originais,
`Scaler::standardize()`, e so entao `predict()`.

O teste da celula seguinte compila o header aqui no Colab e compara com o scikit-learn.
Se falhar, nao trate os artefatos como equivalentes.

In [ ]:
# Folhas puras: cada folha vota numa classe so. E o que o micromlgen assume ao gerar
# votes[argmax]. Se houver folha mista, o C++ pode decidir diferente do Python.
for arvore in floresta.estimators_:
    folhas = arvore.tree_.children_left == -1
    contagens = arvore.tree_.value[folhas, 0, :]
    assert (np.count_nonzero(contagens, axis=1) == 1).all(), \
        "Folhas mistas: inspecione amostras identicas com targets diferentes. Nao altere rotulos so para exportar."

Path("AIoTRandomForest_micromlgen.hpp").write_text(port(floresta), encoding="utf-8")

# O scaler, no mesmo formato do app30: Scaler::standardize(entrada, saida).
media = escalador.mean_.astype(np.float32)
escala = escalador.scale_.astype(np.float32)
cabecalho = '''#pragma once

// StandardScaler ajustado no treino do app19 (notebook da Random Forest).
// Ordem: temperatura, umidade, delta_distancia, luz, mov_max, incl_max.
namespace Scaler {
    const float media[6] = {__MEDIA__};
    const float escala[6] = {__ESCALA__};

    void standardize(const float entrada[6], float saida[6]) {
        for (int i = 0; i < 6; i++) saida[i] = (entrada[i] - media[i]) / escala[i];
    }
}
'''.replace("__MEDIA__", ", ".join(f"{v:.9e}f" for v in media)) \
   .replace("__ESCALA__", ", ".join(f"{v:.9e}f" for v in escala))
Path("AIoTSmartBagScaler.hpp").write_text(cabecalho, encoding="utf-8")

# O C++ recebe os valores JA normalizados, que e o que o ESP32 vai entregar.
X_teste_norm = escalador.transform(X_teste).astype(np.float32)
pd.DataFrame(X_teste_norm, columns=FEATURES).to_csv("entradas_teste_normalizadas.csv", index=False)
np.savetxt("classes_teste.csv", predicoes, fmt="%d", header="classe", comments="")
print("media: ", np.round(media, 4))
print("escala:", np.round(escala, 4))
print("Confira: estes dois vetores devem sair IDENTICOS no notebook da MLP.")

In [ ]:
codigo = r"""
#include <iostream>
#include <cstdint>
#include "AIoTRandomForest_micromlgen.hpp"
int main() {
    Eloquent::ML::Port::RandomForest modelo;
    float x[6];
    while (std::cin >> x[0] >> x[1] >> x[2] >> x[3] >> x[4] >> x[5])
        std::cout << modelo.predict(x) << "\n";
}
"""
Path("conferir.cpp").write_text(codigo, encoding="utf-8")
subprocess.run(["g++", "-std=c++11", "conferir.cpp", "-o", "conferir"], check=True)

entrada = pd.DataFrame(X_teste_norm).to_csv(index=False, header=False, sep=" ")
saida = subprocess.run(["./conferir"], input=entrada, text=True, capture_output=True, check=True)
classes_cpp = np.fromstring(saida.stdout, sep=" ", dtype=int)
assert np.array_equal(classes_cpp, predicoes), \
    "C++ divergiu do Python: conferir exportador e precisao numerica ANTES de embarcar."
print("Python e C++ concordam nas", len(classes_cpp), "amostras de teste.")

# Empate so seria possivel com numero par de arvores; com 21, nenhuma amostra empata.
print("Arvores:", floresta.n_estimators, "| impar:", floresta.n_estimators % 2 == 1)

## 9. Baixar

O ZIP reune modelo, os dois headers, versoes e amostras de conferencia.

- `modelo_smartbag.pkl` → `app20/api/`, junto com as versoes registradas
- `AIoTRandomForest_micromlgen.hpp` e `AIoTSmartBagScaler.hpp` → `app21/device/src/`

In [ ]:
with zipfile.ZipFile("smartbag_rf.zip", "w") as pacote:
    for nome in ["modelo_smartbag.pkl",
                 "AIoTRandomForest_micromlgen.hpp",
                 "AIoTSmartBagScaler.hpp",
                 "metadados_rf.json",
                 "entradas_teste_normalizadas.csv",
                 "classes_teste.csv"]:
        pacote.write(nome)
files.download("smartbag_rf.zip")